# 📡 Seeing Signals — Part I
## IQ Signal Modeling and Dataset Construction

**Workflow:**
1. Configure local output directory
2. Install dependencies
3. Define modulation constellations (16 schemes)
4. Impairment models (AWGN, Phase Noise, I/Q Imbalance, Jamming)
5. Constellation image renderer
6. Structured label schema
7. Dataset builder
8. SEP vs SNR curves
9. Visual inspection

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path('../data/generated')
(OUTPUT_DIR / 'images').mkdir(parents=True, exist_ok=True)
print(f'Output directory: {OUTPUT_DIR.resolve()}')


## Cell 2 — Install & Imports

In [ ]:
!pip install numpy matplotlib pillow tqdm pandas -q

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image
import json
from tqdm import tqdm
import pandas as pd

np.random.seed(42)
print('Imports OK')

## Cell 3 — Global Dataset Parameters

Dataset generation parameters.
- `SAMPLES_PER_CFG = 20` for a compact run (~31K εικόνες, ~1 ώρα)
- `SAMPLES_PER_CFG = 50` for a larger dataset (~77K εικόνες, ~3-4 ώρες)

In [ ]:
N_SYMBOLS       = 512   # symbols per constellation image
IMG_SIZE        = 224   # pixels (224x224)
SAMPLES_PER_CFG = 20    # images per (modulation × impairment) configuration

# ── SNR levels (dB) ──────────────────────────────────────────────────────────
SNR_LEVELS = {
    'low':    (0,  5),
    'medium': (5, 15),
    'high':  (15, 30),
}

# ── Phase-noise severity (std of wrapped-Gaussian, radians) ──────────────────
PHASE_NOISE_LEVELS = {
    'none':     0.00,
    'mild':     0.05,   # ~3°
    'moderate': 0.15,   # ~9°
    'severe':   0.30,   # ~17°
}

# ── I/Q imbalance: (amplitude ratio α, phase imbalance φ in radians) ─────────
IQ_IMBALANCE_LEVELS = {
    'none':     (1.00, 0.00),
    'mild':     (1.05, 0.03),   # 5% amp, ~1.7°
    'moderate': (1.10, 0.06),   # 10% amp, ~3.4°
    'severe':   (1.20, 0.12),   # 20% amp, ~6.9°
}

# ── Jamming (power relative to signal) ───────────────────────────────────────
JAMMING_LEVELS = {
    'none':    0.0,
    'present': 0.3,
}

total_samples = (16 * len(SNR_LEVELS) * len(PHASE_NOISE_LEVELS) *
                 len(IQ_IMBALANCE_LEVELS) * len(JAMMING_LEVELS) * SAMPLES_PER_CFG)
print(f'Parameters configured')
print(f'Estimated total images: {total_samples:,}')

## Cell 4 — Constellation Point Generators (16 Modulation Schemes)

In [ ]:
def ask_constellation(M):
    """M-ASK: M real symbols on I axis."""
    levels = np.arange(-(M - 1), M, 2, dtype=float)
    return levels + 0j

def bpsk_constellation():
    return np.array([-1.0 + 0j, 1.0 + 0j])

def qpsk_constellation():
    angles = np.array([45, 135, 225, 315]) * np.pi / 180
    return np.exp(1j * angles)

def qam_constellation(M):
    """
    Square QAM για M = 16, 64, 256.
    Rectangular για M = 32, 128.
    """
    sqM = int(np.sqrt(M))
    if sqM * sqM == M:
        levels = np.arange(-(sqM - 1), sqM, 2, dtype=float)
        return np.array([I + 1j*Q for Q in levels for I in levels])
    else:
        cols = int(np.sqrt(M / 2))
        rows = 2 * cols
        I_lev = np.arange(-(cols - 1), cols, 2, dtype=float)
        Q_lev = np.arange(-(rows - 1), rows, 2, dtype=float)
        return np.array([I + 1j*Q for Q in Q_lev for I in I_lev])

def hqam_constellation(M):
    """
    Hexagonal QAM: offset rows για hex-grid layout.
    M = 4: diamond, M = 16/64: hex-offset grid.
    """
    if M == 4:
        return np.array([1+0j, -1+0j, 0+1j, 0-1j])
    sqM = int(np.sqrt(M))
    points = []
    for row in range(sqM):
        offset = 0.5 if row % 2 else 0.0
        for col in range(sqM):
            points.append(
                (col * 2 + offset - (sqM - 1 + offset)) +
                1j * (row * np.sqrt(3) - (sqM - 1) * np.sqrt(3) / 2)
            )
    return np.array(points[:M])

def apsk_constellation(M):
    """
    M-APSK: ομόκεντροι κύκλοι (DVB-S2 convention).
    """
    configs = {
        16:  [(4, 1.0), (12, 2.70)],
        32:  [(4, 1.0), (12, 2.53), (16, 4.52)],
        64:  [(4, 1.0), (12, 2.40), (20, 4.30), (28, 6.50)],
        128: [(4, 1.0), (12, 2.30), (20, 4.10), (28, 6.20), (64, 9.00)],
    }
    points = []
    for n_pts, radius in configs[M]:
        angles = np.linspace(0, 2*np.pi, n_pts, endpoint=False)
        points.extend(radius * np.exp(1j * angles))
    return np.array(points[:M])

# ── Registry ─────────────────────────────────────────────────────────────────
MODULATION_REGISTRY = {
    '4ASK':    ask_constellation(4),
    '8ASK':    ask_constellation(8),
    'BPSK':    bpsk_constellation(),
    'QPSK':    qpsk_constellation(),
    '4HQAM':   hqam_constellation(4),
    '16HQAM':  hqam_constellation(16),
    '64HQAM':  hqam_constellation(64),
    '16QAM':   qam_constellation(16),
    '32QAM':   qam_constellation(32),
    '64QAM':   qam_constellation(64),
    '128QAM':  qam_constellation(128),
    '256QAM':  qam_constellation(256),
    '16APSK':  apsk_constellation(16),
    '32APSK':  apsk_constellation(32),
    '64APSK':  apsk_constellation(64),
    '128APSK': apsk_constellation(128),
}

print(f'Modulation schemes ({len(MODULATION_REGISTRY)}):')
for name, pts in MODULATION_REGISTRY.items():
    print(f'  {name:10s}: {len(pts)} points')

## Cell 5 — Impairment Models

In [ ]:
def generate_symbols(constellation, n):
    """Randomly sample n symbols from the constellation."""
    idx = np.random.randint(0, len(constellation), size=n)
    return constellation[idx].copy()

def normalize_energy(symbols):
    """Normalize average symbol energy to 1."""
    avg_power = np.mean(np.abs(symbols) ** 2)
    return symbols / np.sqrt(avg_power) if avg_power > 0 else symbols

def add_awgn(symbols, snr_db):
    """
    AWGN: E_s/N_0 = snr_db (dB).
    Noise variance: σ² = 1 / (2 * SNR_linear)
    """
    snr_lin  = 10 ** (snr_db / 10.0)
    noise_std = 1.0 / np.sqrt(2 * snr_lin)
    noise = noise_std * (np.random.randn(len(symbols)) + 1j * np.random.randn(len(symbols)))
    return symbols + noise

def add_phase_noise(symbols, sigma_rad):
    """
    Phase noise ως wrapped Gaussian: θ ~ N(0, σ²).
    Κάθε σύμβολο πολλαπλασιάζεται με e^{jθ}.
    """
    if sigma_rad == 0.0:
        return symbols
    theta = np.random.normal(0, sigma_rad, size=len(symbols))
    return symbols * np.exp(1j * theta)

def add_iq_imbalance(symbols, alpha, phi):
    """
    I/Q imbalance analytical model:
        r = c1 * s + c2 * conj(s)
    όπου c1 = 0.5*(1 + α*e^{jφ}), c2 = 0.5*(1 - α*e^{-jφ})
    alpha: amplitude imbalance ratio
    phi:   phase imbalance (rad)
    """
    if alpha == 1.0 and phi == 0.0:
        return symbols
    c1 = 0.5 * (1 + alpha * np.exp( 1j * phi))
    c2 = 0.5 * (1 - alpha * np.exp(-1j * phi))
    return c1 * symbols + c2 * np.conj(symbols)

def add_jamming(symbols, jammer_power):
    """
    Narrowband jamming: single tone σε τυχαίο frequency offset.
    jammer_power: ισχύς jamming σχετικά με normalized signal.
    """
    if jammer_power == 0.0:
        return symbols
    amp = np.sqrt(jammer_power)
    freq_offset = np.random.uniform(-0.1, 0.1)
    t = np.arange(len(symbols))
    tone = amp * np.exp(1j * 2 * np.pi * freq_offset * t)
    return symbols + tone

def apply_impairments(symbols, snr_db, phase_sigma, iq_alpha, iq_phi, jam_power):
    """
    Impairment order in the transceiver:
    Normalize → IQ imbalance → Phase noise → Jamming → AWGN
    """
    s = normalize_energy(symbols)
    s = add_iq_imbalance(s, iq_alpha, iq_phi)
    s = add_phase_noise(s, phase_sigma)
    s = add_jamming(s, jam_power)
    s = add_awgn(s, snr_db)
    return s

# Sanity check
_syms = generate_symbols(MODULATION_REGISTRY['16QAM'], 256)
_rx   = apply_impairments(_syms, snr_db=15, phase_sigma=0.1,
                           iq_alpha=1.05, iq_phi=0.03, jam_power=0.0)
print('Impairment pipeline OK — shape:', _rx.shape)

## Cell 6 — Constellation Image Renderer

In [ ]:
def render_constellation(symbols, img_size=IMG_SIZE, margin=0.05):
    """
    IQ samples → PIL Image (RGB, img_size×img_size).
    Scatter plot with a black background and white points.
    """
    I = symbols.real
    Q = symbols.imag

    max_val = max(np.max(np.abs(I)), np.max(np.abs(Q))) + 1e-6
    lim = max_val * (1 + margin)

    fig, ax = plt.subplots(figsize=(2.24, 2.24), dpi=100)
    ax.scatter(I, Q, s=3, c='white', alpha=0.85, linewidths=0)
    ax.set_xlim(-lim, lim)
    ax.set_ylim(-lim, lim)
    ax.set_facecolor('black')
    fig.patch.set_facecolor('black')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_aspect('equal')
    plt.tight_layout(pad=0)

    fig.canvas.draw()
    width, height = fig.canvas.get_width_height()
    buf = np.frombuffer(fig.canvas.tostring_rgb(), dtype=np.uint8)
    buf = buf.reshape(height, width, 3)
    plt.close(fig)

    img = Image.fromarray(buf).resize((img_size, img_size), Image.LANCZOS)
    return img

# Test render
test_syms = generate_symbols(MODULATION_REGISTRY['16QAM'], N_SYMBOLS)
test_rx   = apply_impairments(test_syms, 15, 0.0, 1.0, 0.0, 0.0)
test_img  = render_constellation(test_rx)

plt.figure(figsize=(3,3), facecolor='black')
plt.imshow(test_img)
plt.title('16-QAM test (SNR=15dB, no impairments)', color='white', fontsize=9)
plt.axis('off')
plt.tight_layout()
plt.show()
print('Renderer OK — image size:', test_img.size)

## Cell 7 — Structured Label Schema

In [ ]:
def make_label(mod_name, snr_key, pn_key, iq_key, jam_key, snr_db):
    """
    Structured label for each sample.
    Ερωτήσεις που απαντά:
      - Ποιο modulation χρησιμοποιείται;
      - Υπάρχει phase noise; Πόσο severe;
      - Υπάρχει I/Q imbalance; Πόσο severe;
      - Υπάρχει jamming;
      - Ποιο είναι το SNR range;
    """
    return {
        'modulation':        mod_name,
        'snr_range':         snr_key,
        'snr_db':            round(float(snr_db), 2),
        'phase_noise':       pn_key,
        'iq_imbalance':      iq_key,
        'jamming':           jam_key,
        'has_phase_noise':   pn_key  != 'none',
        'has_iq_imbalance':  iq_key  != 'none',
        'has_jamming':       jam_key != 'none',
    }

# VLM question templates (used in Part II)
VLM_QUESTIONS = [
    'What modulation scheme is used in this constellation?',
    'Is there phase noise? If yes, what is its severity?',
    'Is there I/Q imbalance? If yes, what is its severity?',
    'Is there external interference or jamming?',
    'What is the SNR range of this signal?',
]

# Example label
sample_label = make_label('16QAM', 'medium', 'mild', 'none', 'none', 12.5)
print('Example label:')
for k, v in sample_label.items():
    print(f'  {k:20s}: {v}')

## Cell 8 — Dataset Builder

⚠️ **Αυτό το κελί τρέχει για αρκετή ώρα** ανάλογα με το `SAMPLES_PER_CFG`.
Μην το διακόψεις — αποθηκεύει στο Google Drive.

In [ ]:
def build_dataset():
    records = []
    sample_id = 0

    total = (len(MODULATION_REGISTRY) *
             len(SNR_LEVELS) *
             len(PHASE_NOISE_LEVELS) *
             len(IQ_IMBALANCE_LEVELS) *
             len(JAMMING_LEVELS) *
             SAMPLES_PER_CFG)

    with tqdm(total=total, desc='Generating dataset') as pbar:
        for mod_name, constellation in MODULATION_REGISTRY.items():
            for snr_key, (snr_lo, snr_hi) in SNR_LEVELS.items():
                for pn_key, pn_sigma in PHASE_NOISE_LEVELS.items():
                    for iq_key, (iq_alpha, iq_phi) in IQ_IMBALANCE_LEVELS.items():
                        for jam_key, jam_power in JAMMING_LEVELS.items():
                            for _ in range(SAMPLES_PER_CFG):

                                snr_db = np.random.uniform(snr_lo, snr_hi)

                                raw = generate_symbols(constellation, N_SYMBOLS)
                                rx  = apply_impairments(raw, snr_db, pn_sigma,
                                                        iq_alpha, iq_phi, jam_power)

                                img   = render_constellation(rx, IMG_SIZE)
                                fname = f'{sample_id:06d}.png'
                                img.save(OUTPUT_DIR / 'images' / fname)

                                label = make_label(mod_name, snr_key, pn_key,
                                                   iq_key, jam_key, snr_db)
                                label['sample_id'] = sample_id
                                label['filename']  = fname
                                records.append(label)

                                sample_id += 1
                                pbar.update(1)

    # Save labels
    df = pd.DataFrame(records)
    df.to_csv(OUTPUT_DIR / 'labels.csv', index=False)
    with open(OUTPUT_DIR / 'labels.json', 'w') as f:
        json.dump(records, f, indent=2)

    print(f'\nDataset complete: {sample_id:,} samples')
    print(f'Saved to: {OUTPUT_DIR}')
    return df

df = build_dataset()

## Cell 9 — Dataset Statistics

In [ ]:
print('=== Dataset Summary ===')
print(f'Total samples       : {len(df):,}')
print(f'Modulation classes  : {df["modulation"].nunique()}')
print(f'SNR ranges          : {df["snr_range"].unique().tolist()}')
print(f'Phase noise levels  : {df["phase_noise"].unique().tolist()}')
print(f'I/Q imbalance levels: {df["iq_imbalance"].unique().tolist()}')
print(f'Jamming labels      : {df["jamming"].unique().tolist()}')
print(f'\nSamples per modulation:')
print(df['modulation'].value_counts().to_string())
print(f'\nColumns: {list(df.columns)}')

## Cell 10 — SEP vs SNR Curves (Phase Noise = Wrapped Gaussian)

In [ ]:
def nearest_neighbor_decision(rx_symbols, constellation):
    """Hard decision using the nearest constellation point."""
    return np.array([
        constellation[np.argmin(np.abs(r - constellation))]
        for r in rx_symbols
    ])

def estimate_sep(constellation, snr_db, phase_sigma, n_trials=5000):
    """Monte Carlo SEP estimate with phase noise and AWGN only."""
    raw  = generate_symbols(constellation, n_trials)
    raw  = normalize_energy(raw)
    rx   = add_phase_noise(raw, phase_sigma)
    rx   = add_awgn(rx, snr_db)
    norm_const = normalize_energy(constellation)
    dec  = nearest_neighbor_decision(rx, norm_const)
    ref  = normalize_energy(raw)
    sep  = np.mean(np.abs(dec - ref) > 1e-6)
    return sep

def plot_sep_vs_snr():
    qam_orders  = [16, 64, 256]
    snr_range   = np.arange(0, 32, 2)
    pn_sigmas   = [0.00, 0.05, 0.15, 0.30]
    pn_labels   = ['No PN', 'Mild (σ=0.05)', 'Moderate (σ=0.15)', 'Severe (σ=0.30)']
    colors      = ['#2ecc71', '#f39c12', '#e74c3c', '#9b59b6']
    linestyles  = ['-', '--', '-.', ':']

    fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor='#0d0d0d')
    fig.suptitle('SEP vs SNR — QAM under Phase Noise (Wrapped Gaussian)',
                 color='white', fontsize=13, y=1.02)

    for ax, M in zip(axes, qam_orders):
        const = qam_constellation(M)
        ax.set_facecolor('#1a1a2e')
        ax.set_title(f'{M}-QAM', color='white', fontsize=12)
        ax.set_xlabel('SNR (dB)', color='#aaaaaa')
        ax.set_ylabel('SEP', color='#aaaaaa')
        ax.tick_params(colors='#aaaaaa')
        ax.set_yscale('log')
        ax.set_ylim(1e-4, 1.1)
        ax.grid(True, color='#333355', linestyle='--', alpha=0.6)
        for spine in ax.spines.values():
            spine.set_edgecolor('#333355')

        for sigma, label, color, ls in zip(pn_sigmas, pn_labels, colors, linestyles):
            print(f'  Computing {M}-QAM, {label}...')
            sep_vals = [estimate_sep(const, snr, sigma) for snr in snr_range]
            ax.plot(snr_range, sep_vals, color=color, linestyle=ls,
                    linewidth=1.8, label=label, marker='o', markersize=3)

        ax.legend(fontsize=8, facecolor='#0d0d0d', labelcolor='white',
                  edgecolor='#444444')

    plt.tight_layout()
    sep_path = OUTPUT_DIR / 'sep_vs_snr.png'
    plt.savefig(sep_path, dpi=150, bbox_inches='tight', facecolor='#0d0d0d')
    plt.show()
    print(f'\nSEP plot saved → {sep_path}')

plot_sep_vs_snr()

## Cell 11 — Visual Inspection Grid

In [ ]:
def show_sample_grid(n_cols=4):
    mods = list(MODULATION_REGISTRY.keys())
    n_rows = int(np.ceil(len(mods) / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols,
                              figsize=(n_cols * 3.5, n_rows * 3.5),
                              facecolor='#0d0d0d')
    fig.suptitle('Constellation Images — medium SNR, no impairments',
                 color='white', fontsize=13)

    for ax, mod in zip(axes.flat, mods):
        subset = df[
            (df['modulation']  == mod) &
            (df['snr_range']   == 'medium') &
            (df['phase_noise'] == 'none') &
            (df['iq_imbalance']== 'none') &
            (df['jamming']     == 'none')
        ]
        if len(subset) == 0:
            ax.axis('off')
            continue
        fname = subset.iloc[0]['filename']
        img   = Image.open(OUTPUT_DIR / 'images' / fname)
        ax.imshow(img)
        ax.set_title(mod, color='white', fontsize=10, pad=4)
        ax.axis('off')

    # Hide unused subplots
    for ax in axes.flat[len(mods):]:
        ax.axis('off')

    plt.tight_layout()
    grid_path = OUTPUT_DIR / 'sample_grid.png'
    plt.savefig(grid_path, dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
    plt.show()
    print(f'Sample grid saved → {grid_path}')

show_sample_grid()

## Cell 12 — Impairment Effect Visualization

Visualizes how each impairment affects a 16-QAM constellation.

In [ ]:
def show_impairment_effects(mod='16QAM', snr_db=20):
    constellation = MODULATION_REGISTRY[mod]

    configs = [
        ('Clean (SNR=20dB)',          0.00, 1.00, 0.00, 0.0),
        ('Mild Phase Noise',          0.05, 1.00, 0.00, 0.0),
        ('Severe Phase Noise',        0.30, 1.00, 0.00, 0.0),
        ('Mild I/Q Imbalance',        0.00, 1.05, 0.03, 0.0),
        ('Severe I/Q Imbalance',      0.00, 1.20, 0.12, 0.0),
        ('Jamming',                   0.00, 1.00, 0.00, 0.3),
        ('PN + IQ + Jamming (mild)',  0.05, 1.05, 0.03, 0.1),
        ('Low SNR (5dB)',             0.00, 1.00, 0.00, 0.0),
    ]
    snr_values = [snr_db]*7 + [5]

    fig, axes = plt.subplots(2, 4, figsize=(16, 8), facecolor='#0d0d0d')
    fig.suptitle(f'Impairment Effects on {mod}', color='white', fontsize=13)

    for ax, (title, pn, alpha, phi, jam), snr in zip(axes.flat, configs, snr_values):
        raw = generate_symbols(constellation, N_SYMBOLS)
        rx  = apply_impairments(raw, snr, pn, alpha, phi, jam)
        img = render_constellation(rx)
        ax.imshow(img)
        ax.set_title(title, color='white', fontsize=8.5, pad=4)
        ax.axis('off')

    plt.tight_layout()
    eff_path = OUTPUT_DIR / 'impairment_effects.png'
    plt.savefig(eff_path, dpi=120, bbox_inches='tight', facecolor='#0d0d0d')
    plt.show()
    print(f'Impairment effects saved → {eff_path}')

show_impairment_effects()